<a href="https://colab.research.google.com/github/massamba0809/churn-prediction-ml/blob/partie-2-eda/notebooks/02_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Partie 2 — Analyse exploratoire des données (EDA)

**Projet :** prédiction du churn client (groupe de 5)
**Responsable :** Sine Ndoye (Membre 2)
**Dataset :** Telco Customer Churn (`data/raw/telco_customer_churn.csv`)

Objectif : comprendre la structure des données, les distributions des variables
et leurs liens avec le churn, afin de guider le prétraitement (Membre 3).

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Le CSV n'est pas au même endroit selon qu'on exécute sur PC (dépôt) ou sur Colab
chemins_possibles = [
    Path("../data/raw/telco_customer_churn.csv"),   # depuis le dossier notebooks/
    Path("data/raw/telco_customer_churn.csv"),      # depuis la racine du dépôt
    Path("telco_customer_churn.csv"),               # Colab (fichier importé)
]
CHEMIN_DONNEES = next((p for p in chemins_possibles if p.exists()), None)
if CHEMIN_DONNEES is None:
    raise FileNotFoundError("CSV introuvable : importe telco_customer_churn.csv")

# Dossier où on sauvegardera les graphiques
RACINE = CHEMIN_DONNEES.parents[2] if "raw" in CHEMIN_DONNEES.parts else Path(".")
DOSSIER_FIGURES = RACINE / "reports" / "figures"
DOSSIER_FIGURES.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CHEMIN_DONNEES)
print("Fichier chargé :", CHEMIN_DONNEES)
print("Dimensions :", df.shape)

Fichier chargé : telco_customer_churn.csv
Dimensions : (7043, 21)


In [3]:
# Copie de travail : df (brut) reste intact
eda = df.copy()

# TotalCharges est lu comme du texte à cause de 11 valeurs " " (cf. Partie 1)
eda["TotalCharges"] = pd.to_numeric(eda["TotalCharges"], errors="coerce")

# Version numérique de la cible : 1 = parti, 0 = resté
eda["Churn_num"] = (eda["Churn"] == "Yes").astype(int)

print("Type de TotalCharges :", eda["TotalCharges"].dtype)
print("Valeurs manquantes dans TotalCharges :", eda["TotalCharges"].isna().sum())
print("Taux de churn global :", round(eda["Churn_num"].mean() * 100, 2), "%")

Type de TotalCharges : float64
Valeurs manquantes dans TotalCharges : 11
Taux de churn global : 26.54 %


**Observation :** le dataset compte 7 043 clients et 21 colonnes. `TotalCharges`
a été converti en nombre sur une copie de travail : 11 valeurs vides (clients avec
`tenure = 0`) deviennent `NaN`. Elles sont laissées telles quelles dans l'EDA ;
leur traitement est confié au prétraitement (Membre 3). Le taux de churn global
est de **26,5 %**.

## 1. Rappel de la structure et de la qualité des données

La Partie 1 a déjà décrit chaque variable en détail. Ici, on se limite à un rappel
et on classe les variables par type pour organiser l'analyse.

In [4]:
# Variables numériques continues
VAR_NUM = ["tenure", "MonthlyCharges", "TotalCharges"]

# Variables catégorielles : tout le reste, sauf l'identifiant et la cible
VAR_CAT = [c for c in eda.columns if c not in VAR_NUM + ["customerID", "Churn", "Churn_num"]]

print(f"{len(VAR_NUM)} variables numériques : {VAR_NUM}")
print(f"{len(VAR_CAT)} variables catégorielles : {VAR_CAT}")

3 variables numériques : ['tenure', 'MonthlyCharges', 'TotalCharges']
16 variables catégorielles : ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [5]:
print("Doublons :", df.duplicated().sum())
print("Valeurs manquantes par colonne :")
print(eda.isna().sum()[eda.isna().sum() > 0])

eda[VAR_NUM].describe().round(2)

Doublons : 0
Valeurs manquantes par colonne :
TotalCharges    11
dtype: int64


,tenure,MonthlyCharges,TotalCharges
count,7043.00,7043.00,7032.00
mean,32.37,64.76,2283.30
std,24.56,30.09,2266.77
min,0.00,18.25,18.80
25%,9.00,35.50,401.45
50%,29.00,70.35,1397.48
75%,55.00,89.85,3794.74
max,72.00,118.75,8684.80


**Observations :**
- 3 variables numériques et 16 catégorielles (hors identifiant et cible).
- Aucun doublon ; seules 11 valeurs manquent, dans `TotalCharges`.
- `tenure` (ancienneté en mois) va de 0 à 72 : la moitié des clients ont moins de 29 mois.
- `MonthlyCharges` va de 18 à 119 $/mois.
- `TotalCharges` a une moyenne (2 283) bien supérieure à sa médiane (1 397) :
  sa distribution est **asymétrique**, avec beaucoup de petites valeurs et quelques
  très grandes. On le vérifiera visuellement en section 2.